
# 🧠 RSNA Knee Abnormality 2026: The 2.5D DICOM Masterclass
***

## 📑 Overview
Welcome to the RSNA Knee Abnormality Detection challenge! Working with medical DICOM files can be incredibly frustrating. A knee MRI isn't just one image; it's a 3D volume of slices across multiple planes (Sagittal, Coronal, Axial). 

Many top competitors are using a **2.5D Reader** approach. But what does that actually mean? 

In this notebook, we aren't just dumping a PyTorch model on you. We are going to visually deconstruct the 2.5D technique step-by-step. By the end of this masterclass, you will have a plug-and-play PyTorch pipeline that safely loads DICOMs, stacks adjacent slices for spatial awareness, and feeds them into a CNN backbone.

If this visual breakdown helps you understand medical imaging pipelines, **please consider leaving an UPVOTE!** 🌟



## 1. Libraries & Setup
We need `pydicom` to read the medical files and `torch` to build our custom 2.5D datasets.


In [ ]:

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

# Medical Imaging
import pydicom

# Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="darkgrid")



## 2. What exactly is a 2.5D Model?
A standard MRI scan of a knee contains dozens of slices (cross-sections). 
* **2D Approach:** Treat every slice as an independent picture. (Problem: Loses 3D context. A tear might only be visible across 3 consecutive slices).
* **3D Approach:** Feed the entire 3D volume into a 3D-CNN. (Problem: Extremely heavy on GPU memory, slow training, prone to overfitting).

**The 2.5D Solution:** 
We take a target slice, plus $N$ slices before it and $N$ slices after it, and stack them into the **channels** of our image tensor. 
Instead of a standard RGB image `(3, Height, Width)`, we feed our model a stacked tensor `(Channels, Height, Width)` where each channel is a sequential MRI slice. This gives the CNN local spatial awareness without the heavy cost of 3D convolutions.



## 3. Extracting and Visualizing DICOMs
Let's look at the actual data. We will locate a single patient's MRI study, sort the slices by their spatial position (Instance Number), and plot them.


In [ ]:

DATA_DIR = '/kaggle/input/rsna-knee-abnormality-detection'
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, 'train_images')

# Let's find a random patient and one of their series (e.g., Sagittal T2/STIR)
try:
    patient_ids = os.listdir(TRAIN_IMAGES_DIR)
    sample_patient = patient_ids[0]
    sample_series = os.listdir(os.path.join(TRAIN_IMAGES_DIR, sample_patient))[0]
    series_path = os.path.join(TRAIN_IMAGES_DIR, sample_patient, sample_series)
    
    dicom_files = glob.glob(f"{series_path}/*.dcm")
    
    # Sort files properly by InstanceNumber (their actual physical order in the MRI machine)
    def get_instance_number(filepath):
        ds = pydicom.dcmread(filepath, stop_before_pixels=True)
        return int(ds.InstanceNumber)

    dicom_files = sorted(dicom_files, key=get_instance_number)
    
    print(f"Loaded Patient: {sample_patient} | Series: {sample_series}")
    print(f"Total slices in this series: {len(dicom_files)}")

except Exception as e:
    print("Run this in the Kaggle environment with the dataset attached to see the outputs!")


In [ ]:

# Visualizing a sequence of 5 slices (A mini 2.5D stack)
def plot_dicom_sequence(file_list, start_idx=10, num_slices=5):
    if len(file_list) == 0: return
    
    fig, axes = plt.subplots(1, num_slices, figsize=(20, 5))
    fig.suptitle('Sequential MRI Slices (Building our 2.5D Stack)', fontsize=16)
    
    for i in range(num_slices):
        idx = start_idx + i
        if idx < len(file_list):
            ds = pydicom.dcmread(file_list[idx])
            img = ds.pixel_array
            
            # Normalize to 0-255 for visualization
            img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-6)
            img = (img * 255).astype(np.uint8)
            
            axes[i].imshow(img, cmap='bone')
            axes[i].set_title(f"Slice {idx} (Z-pos: {ds.SliceLocation:.1f})")
            axes[i].axis('off')
            
    plt.tight_layout()
    plt.show()

try:
    plot_dicom_sequence(dicom_files, start_idx=len(dicom_files)//2 - 2, num_slices=5)
except Exception as e:
    pass



## 4. Building the 2.5D PyTorch Dataset
This is the core of the notebook. We will write a PyTorch `Dataset` that takes a central slice and stacks `N` neighboring slices into a single tensor.


In [ ]:

class RSNA25DDataset(Dataset):
    def __init__(self, df, image_dir, transform=None, num_channels=5):
        self.df = df
        self.image_dir = image_dir
        self.transform = transform
        self.num_channels = num_channels
        
    def __len__(self):
        return len(self.df)
    
    def read_dicom(self, path):
        ds = pydicom.dcmread(path)
        img = ds.pixel_array.astype(np.float32)
        # Standardize
        img = (img - np.min(img)) / (np.max(img) - np.min(img) + 1e-6)
        img = cv2.resize(img, (224, 224))
        return img
        
    def __getitem__(self, idx):
        # In a real run, this fetches the patient and series ID
        row = self.df.iloc[idx]
        patient_id = row['patient_id']
        series_id = row['series_id']
        
        series_path = os.path.join(self.image_dir, str(patient_id), str(series_id))
        dicom_files = sorted(glob.glob(f"{series_path}/*.dcm"))
        
        # We take the middle slices of the MRI as our focal point
        center_idx = len(dicom_files) // 2
        half_channels = self.num_channels // 2
        
        start_idx = max(0, center_idx - half_channels)
        end_idx = min(len(dicom_files), center_idx + half_channels + 1)
        
        selected_files = dicom_files[start_idx:end_idx]
        
        # Read and stack slices into channels
        images = [self.read_dicom(f) for f in selected_files]
        
        # Pad with zeros if we are at the edge and don't have enough slices
        while len(images) < self.num_channels:
            images.append(np.zeros((224, 224), dtype=np.float32))
            
        # Shape: (Channels, Height, Width)
        image_tensor = np.stack(images, axis=0)
        image_tensor = torch.tensor(image_tensor, dtype=torch.float32)
        
        label = torch.tensor(row['target'], dtype=torch.float32) # Assumes 'target' column
        
        return image_tensor, label

print("2.5D Dataset Class ready.")



## 5. Adapting a CNN for 2.5D (N-Channels)
Pre-trained models like ResNet expect 3 channels (RGB). Our 2.5D tensor has `num_channels` (e.g., 5). We modify the very first convolutional layer to accept our custom depth.


In [ ]:

class RSNA25DModel(nn.Module):
    def __init__(self, num_classes=1, in_channels=5):
        super(RSNA25DModel, self).__init__()
        
        # Load a pre-trained ResNet34
        self.backbone = models.resnet34(pretrained=True)
        
        # Modify the first layer to accept 'in_channels' instead of 3
        original_conv1 = self.backbone.conv1
        self.backbone.conv1 = nn.Conv2d(
            in_channels=in_channels, 
            out_channels=original_conv1.out_channels, 
            kernel_size=original_conv1.kernel_size, 
            stride=original_conv1.stride, 
            padding=original_conv1.padding, 
            bias=original_conv1.bias
        )
        
        # Copy weights from the original 3 channels and average them for the new channels
        with torch.no_grad():
            self.backbone.conv1.weight.copy_(
                original_conv1.weight.mean(dim=1, keepdim=True).repeat(1, in_channels, 1, 1)
            )
            
        # Modify the final classification head
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes)
        )
        
    def forward(self, x):
        return self.backbone(x)

# Initialize model to accept 5 stacked slices
model = RSNA25DModel(num_classes=1, in_channels=5)
print(model.backbone.conv1)
print("\nModel successfully adapted for 2.5D inputs!")



## 🚀 Next Steps & Training
You now have the full architectural foundation for a competitive 2.5D pipeline! 

To turn this into a submission:
1. Load `train.csv` into a Pandas DataFrame.
2. Feed it into `RSNA25DDataset`.
3. Train `RSNA25DModel` using `BCEWithLogitsLoss`.

If this visualization and breakdown saved you hours of deciphering DICOM files, **please consider dropping an UPVOTE!** I will be updating this notebook with the full training loop in Version 2. Happy modeling!
